# Fuel Invoice Pre-processing
This test file uses the Textract API to extract relevant details from receipts/invoices. The testing has been done on one file and pandas library has been used for faster delivery as it is a small file.


Importing all libraries and initialising the bucket and document to be processed


In [9]:
import json
import boto3
import io

bucket = 'cn01-project-input-205096516800-us-east-2-an'
document = 'FuelInvoicesTest.pdf'


*analyze_expense* textract library has been used to detect and extract the details needed for the use case. The output consists of the extracted data along with confidence scores. For the purpose of illustration, focus has been laid on printing the extracted data only.

In [8]:
s3_connection = boto3.resource('s3')
client = boto3.client('textract', region_name='us-east-2')

response = client.analyze_expense(
    Document={
        
        'S3Object': {
            'Bucket': bucket,
            'Name': document
        }
    })


data = json.dumps(response, indent=4)
# print(data)
obj = []
for expense_doc in response["ExpenseDocuments"]:
    obj_data = {}
    for line_item_group in expense_doc["SummaryFields"]:        
        if  "LabelDetection" in line_item_group:
            key = line_item_group['LabelDetection']['Text']
        else:
            key = line_item_group['Type']['Text']
        value = line_item_group['ValueDetection']['Text']
        obj_data.update({key:value})

    obj.append(obj_data)

# print(obj)

Printing the output in a json format for easier understanding

In [10]:
print(json.dumps(obj, indent=2))

[
  {
    "ADDRESS": "UFA Hanna\n685 - 1st Avenue West\nHanna, AB\nTBJ 1P#",
    "STREET": "685 - 1st Avenue West",
    "CITY": "Hanna,",
    "STATE": "AB",
    "ZIP_CODE": "TBJ 1P#",
    "NAME": "UFA",
    "ADDRESS_BLOCK": "685 - 1st Avenue West\nHanna, AB\nTBJ 1P#",
    "DATE :": "2022/08/10",
    "TRANS # :": "50746215NSYX",
    "VENDOR_ADDRESS": "UFA Hanna\n685 - 1st Avenue West\nHanna, AB\nTBJ 1P#",
    "VENDOR_NAME": "UFA",
    "VENDOR_PHONE": "(483) 854-4378",
    "VENDOR_URL": "UFA.com/Ratells",
    "SITE ID :": "462",
    "TIME :": "07:46:18",
    "PRODUCT :": "DIESEL",
    "PUMP # :": "15",
    "LITRES :": "266.20",
    "IMPORTANT:": "Retain this copy for\nyour records",
    "Classification:": "Protected A"
  }
]
